# UTS Data Acquisition & Management (INSTIKI)
**Nama / NIM:** Katarina Gita Candranurani_2501010369  |  **Kelas E:** Ganjil / E  |  **Dosen:** Dr. I Gede Totok Suryawan, S.Kom., M.T.  |  **Ujian:** Kamis, 19 November 2026 (open)

## Topik
Sektor **pariwisata**: jumlah perjalanan wisatawan nusantara per provinsi tujuan (BPS), dikaitkan dengan kondisi cuaca (suhu dan curah hujan) di lima provinsi tujuan: Bali, DI Yogyakarta, Jawa Barat, Sumatera Utara, dan Sulawesi Selatan, periode 2022-2025.

## Tiga cara akuisisi
| Cara | Sumber | Dikerjakan di |
|---|---|---|
| (a) Berkas dataset | BPS: Jumlah Perjalanan Wisatawan Nusantara Menurut Provinsi Tujuan (XLSX) | Soal 1 dan 2 |
| (b) API | Open-Meteo Historical Weather API (suhu dan curah hujan harian) | Soal 3 |
| (c) Web scraping | Wikipedia (id): Daftar ibu kota provinsi di Indonesia (halaman statis) | Soal 4 |

Penggabungan (Soal 5): sumber (a) dan (b), kunci `provinsi` + `periode` (tahun-bulan).

## Aturan yang dipatuhi
- Data mentah disimpan di `data/raw` tanpa diubah; setiap sumber punya catatan provenance (URL, tanggal akses, lisensi, MD5) di `docs/provenance.csv`.
- Setiap request web memakai timeout, User-Agent berisi email kampus, jeda antar-request, dan retry.
- Tidak mengambil data dari situs yang dilarang soal; tidak ada API key/token di notebook.

## Bagian 0 - Persiapan

In [49]:
# Buat struktur folder proyek
import os
for d in ["data/raw", "data/processed", "docs", "docs/bukti"]:
    os.makedirs(d, exist_ok=True)      # exist_ok=True: tidak error bila folder sudah ada
print("data :", sorted(os.listdir("data")))
print("docs :", sorted(os.listdir("docs")))

data : ['processed', 'raw']
docs : ['bukti', 'data_ethics_checklist.md', 'provenance.csv']


In [50]:
EMAIL_KAMPUS = "humas@instiki.ac.id"

assert "@" in EMAIL_KAMPUS and "ISI_" not in EMAIL_KAMPUS, "Isi EMAIL_KAMPUS dengan email lengkap, mis. nim@mahasiswa.instiki.ac.id"

with open(".env", "w") as f:
    f.write(f"CAMPUS_EMAIL={EMAIL_KAMPUS}\n")
print(".env tersimpan")

.env tersimpan


In [51]:
# Setup umum: library, aturan request etis, fungsi hash MD5, dan pencatat provenance
import json, time, hashlib, re
import datetime as dt
from pathlib import Path
from io import StringIO
from urllib import robotparser
from urllib.parse import urlparse
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


env = dict(l.strip().split("=", 1) for l in open(".env") if "=" in l)
CAMPUS_EMAIL = env["CAMPUS_EMAIL"]

USER_AGENT = f"UTS-DataAcq-INSTIKI/1.0 (mahasiswa Informatika; {CAMPUS_EMAIL})"
TIMEOUT = 20
DELAY = 2

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})
retry = Retry(total=3, backoff_factor=2,
              status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"])
session.mount("https://", HTTPAdapter(max_retries=retry))

STATUS_ARTI = {200: "OK", 400: "Permintaan salah (cek parameter)", 403: "Ditolak (cek izin/User-Agent)",
               404: "Tidak ditemukan", 429: "Terlalu banyak request", 500: "Server error",
               502: "Bad gateway", 503: "Layanan tidak tersedia", 504: "Gateway timeout"}

def polite_get(url, params=None):
    # GET dengan timeout, penanganan status code, dan jeda sesudahnya
    try:
        resp = session.get(url, params=params, timeout=TIMEOUT)
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"Gagal terhubung ke {url}: {e}")
    print(f"[{resp.status_code}] {STATUS_ARTI.get(resp.status_code, 'status lain')} | {resp.url[:80]}")
    time.sleep(DELAY)              # jeda antar-request
    resp.raise_for_status()        # 4xx/5xx menjadi error yang jelas
    return resp

def md5_file(path):
    # Hash MD5 dari isi file (sidik jari untuk provenance)
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

PROV_FILE = Path("docs/provenance.csv")
def catat_provenance(baris_baru):
    # Tambah/perbarui catatan provenance. Satu baris per berkas (yang terbaru menang),
    # karena respons API memuat waktu proses yang berubah sehingga MD5-nya bisa berbeda tiap unduhan
    lama = pd.read_csv(PROV_FILE) if PROV_FILE.exists() else pd.DataFrame()
    baru = pd.concat([lama, pd.DataFrame(baris_baru)], ignore_index=True)
    baru = baru.drop_duplicates(subset=["file"], keep="last")
    baru.to_csv(PROV_FILE, index=False)
    return baru

print("User-Agent:", USER_AGENT)

User-Agent: UTS-DataAcq-INSTIKI/1.0 (mahasiswa Informatika; humas@instiki.ac.id)


## Soal 1 - Membaca berkas, identifikasi dataset, estimasi storage
Berkas BPS (XLSX) diunduh dari portal BPS pada 12 November 2026, satu berkas per tahun (2022-2025), dan disimpan **tanpa diubah** di `data/raw`.

In [52]:
# Daftar berkas mentah BPS di data/raw
RAW_DIR = Path("data/raw")
files_bps = sorted(RAW_DIR.glob("*.xlsx"))
assert files_bps, "Belum ada file .xlsx di data/raw. Upload berkas BPS (2022-2025) ke folder data/raw dulu."
print(len(files_bps), "berkas BPS:")
for f in files_bps:
    print(" -", f.name, "|", f.stat().st_size, "byte")

4 berkas BPS:
 - Jumlah Perjalanan Wisatawan Nusantara Menurut Provinsi Tujuan, 2022.xlsx | 37018 byte
 - Jumlah Perjalanan Wisatawan Nusantara Menurut Provinsi Tujuan, 2023.xlsx | 37273 byte
 - Jumlah Perjalanan Wisatawan Nusantara Menurut Provinsi Tujuan, 2024.xlsx | 37351 byte
 - Jumlah Perjalanan Wisatawan Nusantara Menurut Provinsi Tujuan, 2025.xlsx | 37442 byte


In [53]:
# Baca setiap berkas dengan parameter yang sesuai struktur file BPS
BULAN = ["Januari","Februari","Maret","April","Mei","Juni","Juli",
         "Agustus","September","Oktober","November","Desember"]

frames = []
tahun_terbaca = []
for f in files_bps:
    # Ambil tahun dari nama berkas (angka 20xx yang terakhir muncul)
    tahun = int(re.findall(r"20\d{2}", f.name)[-1])
    tahun_terbaca.append(tahun)

    # Parameter baca:
    #  sheet_name=0 -> sheet pertama
    #  header=3     -> baris ke-4 menjadi nama kolom (3 baris di atasnya hanya judul)
    #  dtype=str    -> semua dibaca sebagai teks dulu agar tipe tidak salah tebak; diubah ke angka di langkah berikutnya
    #  na_values    -> "-", "--", "NA", dan sel kosong dianggap data hilang
    d = pd.read_excel(f, sheet_name=0, header=3, dtype=str, na_values=["-", "--", "NA", ""])
    d = d.rename(columns={d.columns[0]: "provinsi"})
    d["tahun"] = tahun                # tambah kolom tahun
    d["sumber_file"] = f.name         # catat asal berkas
    frames.append(d)
    print(tahun, "->", d.shape)

assert len(set(tahun_terbaca)) == len(tahun_terbaca), "Ada tahun ganda (berkas duplikat). Hapus salinan dari data/raw lalu jalankan ulang."

2022 -> (39, 16)
2023 -> (39, 16)
2024 -> (39, 16)
2025 -> (39, 16)


In [54]:
# Gabungkan tabel tahunan menjadi satu tabel
df_all = pd.concat(frames, ignore_index=True)

# Buang baris total "INDONESIA" (bukan provinsi) dan baris kosong
df_all = df_all[df_all["provinsi"].notna() & (df_all["provinsi"] != "INDONESIA")].copy()

# Ubah kolom angka dari teks menjadi numerik
kolom_angka = BULAN + ["Tahunan"]
df_all[kolom_angka] = df_all[kolom_angka].apply(pd.to_numeric, errors="coerce")

print(df_all.shape)
print(df_all.groupby("tahun").size())    # harus 38 provinsi per tahun
df_all.head()

(152, 16)
tahun
2022    38
2023    38
2024    38
2025    38
dtype: int64


,provinsi,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,tahun,sumber_file
0,ACEH,526417.0,501651.0,585675.0,434633.0,935025.0,566361.0,748573.0,531072.0,500673.0,513056.0,471520.0,639922.0,6954578.0,2022,Jumlah Perjalanan Wisatawan Nusantara Menurut ...
1,SUMATERA UTARA,2022964.0,1636066.0,1846809.0,1658463.0,2896949.0,1987624.0,2216932.0,1678953.0,1664991.0,1785747.0,1525633.0,2283325.0,23204456.0,2022,Jumlah Perjalanan Wisatawan Nusantara Menurut ...
2,SUMATERA BARAT,926043.0,814741.0,1007630.0,838560.0,1655583.0,988346.0,1176922.0,905311.0,891560.0,949081.0,877671.0,1307815.0,12339263.0,2022,Jumlah Perjalanan Wisatawan Nusantara Menurut ...
3,RIAU,636543.0,581783.0,673534.0,597000.0,897026.0,692293.0,795698.0,662795.0,677309.0,692329.0,640199.0,857727.0,8404236.0,2022,Jumlah Perjalanan Wisatawan Nusantara Menurut ...
4,JAMBI,302504.0,255285.0,313009.0,310853.0,449078.0,316554.0,364514.0,285681.0,280278.0,289153.0,262087.0,351987.0,3780983.0,2022,Jumlah Perjalanan Wisatawan Nusantara Menurut ...


In [55]:
# Baris yang seluruh kolom angkanya kosong
kosong = df_all[df_all[kolom_angka].isna().all(axis=1)]
print(kosong[["provinsi", "tahun"]])

# Apakah provinsi yang sama terisi di tahun lain? (bukti bahwa kosongnya bukan salah baca)
prov_kosong = kosong["provinsi"].unique()
terisi = (df_all[df_all["provinsi"].isin(prov_kosong) & df_all["Tahunan"].notna()]
          .groupby("provinsi")["tahun"].apply(list))
print("\nTahun yang terisi untuk provinsi tersebut:")
print(terisi)

            provinsi  tahun
33  PAPUA BARAT DAYA   2022
35     PAPUA SELATAN   2022
36      PAPUA TENGAH   2022
37  PAPUA PEGUNUNGAN   2022

Tahun yang terisi untuk provinsi tersebut:
provinsi
PAPUA BARAT DAYA    [2023, 2024, 2025]
PAPUA PEGUNUNGAN    [2023, 2024, 2025]
PAPUA SELATAN       [2023, 2024, 2025]
PAPUA TENGAH        [2023, 2024, 2025]
Name: tahun, dtype: object


**Catatan nilai kosong:** baris yang kosong di seluruh bulan adalah provinsi pemekaran Papua pada tahun 2022 (lihat output di atas). Provinsi yang sama terisi pada tahun-tahun berikutnya, sehingga kosongnya berasal dari sumber (BPS belum menyediakan data tahun itu untuk provinsi tersebut), bukan kesalahan pembacaan. Baris dipertahankan dengan nilai `NaN` dan **tidak diisi nilai karangan**.

In [56]:
# --- Identifikasi dataset ---
print("Jenis data     : terstruktur (tabel XLSX)")
print("Jumlah baris   :", df_all.shape[0], "(38 provinsi x", df_all['tahun'].nunique(), "tahun, tanpa baris total)")
print("Jumlah kolom   :", df_all.shape[1])
print("Wilayah        :", df_all['provinsi'].nunique(), "provinsi (seluruh Indonesia)")
print("Cakupan waktu  :", df_all['tahun'].min(), "-", df_all['tahun'].max(), "(data bulanan + total tahunan)")
print("Satuan         : perjalanan (jumlah perjalanan wisatawan nusantara)")
print("\nTipe data:\n", df_all.dtypes)
print("\nNilai kosong per kolom angka:\n", df_all[kolom_angka].isna().sum())

# --- Ukuran berkas ---
total_byte = sum(f.stat().st_size for f in files_bps)
total_mb = total_byte / (1024 * 1024)
print(f"\nUkuran total {len(files_bps)} berkas: {total_byte/1024:.1f} KB ({total_mb:.4f} MB)")

# --- Estimasi storage 1 tahun = ukuran berkas x 12 bulan, ditambah cadangan 30% ---
estimasi_mb = total_mb * 12 * 1.30
print(f"Estimasi 1 tahun: {total_mb:.4f} MB x 12 x 1.30 = {estimasi_mb:.4f} MB")

Jenis data     : terstruktur (tabel XLSX)
Jumlah baris   : 152 (38 provinsi x 4 tahun, tanpa baris total)
Jumlah kolom   : 16
Wilayah        : 38 provinsi (seluruh Indonesia)
Cakupan waktu  : 2022 - 2025 (data bulanan + total tahunan)
Satuan         : perjalanan (jumlah perjalanan wisatawan nusantara)

Tipe data:
 provinsi        object
Januari        float64
Februari       float64
Maret          float64
April          float64
Mei            float64
Juni           float64
Juli           float64
Agustus        float64
September      float64
Oktober        float64
November       float64
Desember       float64
Tahunan        float64
tahun            int64
sumber_file     object
dtype: object

Nilai kosong per kolom angka:
 Januari      4
Februari     4
Maret        4
April        4
Mei          4
Juni         4
Juli         4
Agustus      4
September    4
Oktober      4
November     4
Desember     4
Tahunan      4
dtype: int64

Ukuran total 4 berkas: 145.6 KB (0.1422 MB)
Estimasi 1 tahun:

**Asumsi estimasi storage:** (1) data diunduh ulang setiap bulan; (2) ukuran tiap unduhan kira-kira sama dengan ukuran sekarang (total semua berkas tahunan yang dibaca di atas); (3) berkas disimpan apa adanya tanpa kompresi; (4) cadangan 30% untuk backup, versi lama, dan data turunan. Rumus: ukuran x 12 bulan x 1,3 (angkanya tercetak di output sel di atas).

## Soal 2 - Lisensi, PII / quasi-identifier, Data Ethics Checklist
Pemeriksaan lisensi untuk ketiga sumber, pemindaian kolom PII, dan **Data Ethics Checklist 9 butir** (di bagian akhir notebook, karena butir robots.txt baru terisi setelah Soal 4).

In [57]:
# Provenance berkas BPS: URL, tanggal akses, lisensi, MD5
# >>> ISI: URL halaman tabel BPS (salin dari address bar browser)
URL_BPS = "https://www.bps.go.id/id/statistics-table/2/MjIwMSMy/jumlah-perjalanan-wisatawan-nusantara-menurut-provinsi-tujuan-perjalanan.html"
TANGGAL_AKSES_BPS = "2026-11-12"          # tanggal berkas diunduh
assert URL_BPS.startswith("http") and "ISI_" not in URL_BPS, "Isi URL_BPS dengan URL halaman tabel BPS"

LISENSI_BPS = ("Ketentuan Penggunaan BPS (bukan lisensi CC): gratis, komersial/nonkomersial, boleh salin dan distribusi; "
               "wajib kutip BPS (judul, tanggal akses, tautan). Sumber: https://www.bps.go.id/assets/term-of-use (diakses 2026-11-12)")

# Sitasi sesuai syarat BPS: judul konten, tanggal akses, tautan langsung
SITASI_BPS = ("Badan Pusat Statistik. Jumlah Perjalanan Wisatawan Nusantara Menurut Provinsi Tujuan (Perjalanan), "
              f"2022-2025. Diakses 12 November 2026, {URL_BPS}")
print(SITASI_BPS)

prov = catat_provenance([dict(sumber="BPS", metode="unduh berkas", file=str(f), url=URL_BPS,
                              tanggal_akses=TANGGAL_AKSES_BPS, lisensi=LISENSI_BPS,
                              ukuran_byte=f.stat().st_size, md5=md5_file(f)) for f in files_bps])
print(len(prov), "baris provenance")

Badan Pusat Statistik. Jumlah Perjalanan Wisatawan Nusantara Menurut Provinsi Tujuan (Perjalanan), 2022-2025. Diakses 12 November 2026, https://www.bps.go.id/id/statistics-table/2/MjIwMSMy/jumlah-perjalanan-wisatawan-nusantara-menurut-provinsi-tujuan-perjalanan.html
10 baris provenance


In [58]:
# Simpan bukti lisensi (kutipan) ke docs/bukti. Tambahkan juga screenshot halaman terkait ke folder yang sama.
bukti_bps = """Sumber: https://www.bps.go.id/assets/term-of-use (diakses 2026-11-12)

Kutipan, bagian "Syarat Penggunaan":
"Konten yang tersedia di website ini diberikan kepada Anda secara gratis, di seluruh dunia, terus-menerus, dan non-eksklusif untuk tujuan (i) penggunaan data, baik untuk kepentingan komersial maupun nonkomersial; (ii) penyalinan, pendistribusian, dan/atau pentransmisian konten; dan (iii) pemanfaatan konten dan/atau layanan untuk pengembangan, pengubahan, atau pengekstraksian kepada pihak ketiga dengan cara apa pun."
"memberikan kutipan atau kredit kepada BPS. Kutipan harus mencantumkan judul konten, tanggal akses, penulis (jika ada), dan tautan langsung ke konten asli di website."

Catatan: ketentuan BPS tidak menyebut lisensi Creative Commons; yang berlaku adalah Ketentuan Penggunaan di atas.
"""
bukti_om = """Sumber: https://open-meteo.com/en/terms dan https://open-meteo.com/en/licence (diakses 2026-11-12)

Kutipan singkat:
"You may only use the free API services for non-commercial purposes."
"Less than 10'000 API calls per day, 5'000 per hour and 600 per minute."
Data API dilisensikan di bawah "Attribution 4.0 International (CC BY 4.0)": wajib atribusi (mis. "Weather data by Open-Meteo.com"), tautan ke lisensi, dan tandai bila ada perubahan.

Catatan: tugas kuliah termasuk penggunaan non-komersial/edukasi.
"""
bukti_wp = """Sumber: https://id.wikipedia.org/wiki/Daftar_ibu_kota_provinsi_di_Indonesia
Teks Wikipedia berlisensi CC BY-SA 4.0 (lihat bagian kaki halaman Wikipedia). Atribusi: Wikipedia, "Daftar ibu kota provinsi di Indonesia".
Pemeriksaan robots.txt: lihat docs/bukti/robots_id_wikipedia.txt (dibuat di Soal 4).
"""
for nama, isi in [("lisensi_bps.txt", bukti_bps), ("lisensi_openmeteo.txt", bukti_om), ("lisensi_wikipedia.txt", bukti_wp)]:
    Path("docs/bukti", nama).write_text(isi, encoding="utf-8")
    print("Tersimpan: docs/bukti/" + nama)

Tersimpan: docs/bukti/lisensi_bps.txt
Tersimpan: docs/bukti/lisensi_openmeteo.txt
Tersimpan: docs/bukti/lisensi_wikipedia.txt


In [59]:
# Pindai nama kolom yang berpotensi PII atau quasi-identifier
PII   = ["nama", "nik", "ktp", "email", "telp", "phone", "alamat", "lahir"]
QUASI = ["umur", "usia", "gender", "kelamin", "kode_pos", "desa", "kecamatan", "pekerjaan", "pendidikan"]

temuan = []
for kol in df_all.columns:
    k = str(kol).lower()
    if any(x in k for x in PII):
        temuan.append((kol, "PII (potensial)"))
    elif any(x in k for x in QUASI):
        temuan.append((kol, "Quasi-identifier (potensial)"))

print("Kolom dicek:", list(df_all.columns))
print("Temuan:", temuan if temuan else "tidak ada kolom PII/quasi-identifier")

Kolom dicek: ['provinsi', 'Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni', 'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember', 'Tahunan', 'tahun', 'sumber_file']
Temuan: tidak ada kolom PII/quasi-identifier


**Kesimpulan PII:** tidak ada kolom PII maupun quasi-identifier. Data berupa **agregat jumlah perjalanan per provinsi per bulan** (unit analisis adalah wilayah, bukan orang), sehingga tidak dapat dikaitkan dengan individu. Data cuaca (Open-Meteo) dan tabel ibu kota (Wikipedia) juga tidak memuat data pribadi.

## Soal 3 - Akuisisi melalui API (Open-Meteo)
BPS tidak menyediakan API tanpa pendaftaran key untuk tabel ini, maka dipakai **API publik yang relevan**: Open-Meteo Historical Weather API (tanpa API key) untuk cuaca harian di ibu kota lima provinsi tujuan, periode 2022-2025 (sama dengan data BPS). Request memakai timeout, User-Agent berisi email kampus, jeda, dan retry otomatis untuk status 429/5xx.

In [60]:
# Provinsi (penulisan BPS) -> koordinat ibu kotanya
LOKASI = {
    "BALI":             (-8.65, 115.22),   # Denpasar
    "DI YOGYAKARTA":    (-7.80, 110.36),   # Yogyakarta
    "JAWA BARAT":       (-6.92, 107.61),   # Bandung
    "SUMATERA UTARA":   (3.59, 98.67),     # Medan
    "SULAWESI SELATAN": (-5.15, 119.41),   # Makassar
}
API_URL = "https://archive-api.open-meteo.com/v1/archive"
LISENSI_API = ("CC BY 4.0 (Open-Meteo): wajib atribusi + tautan lisensi; API gratis hanya non-komersial. "
               "Sumber: https://open-meteo.com/en/licence dan https://open-meteo.com/en/terms (diakses 2026-11-12)")

respons = {}       # JSON hasil panggilan per provinsi
rows_api = []      # bahan catatan provenance

for prov_name, (lat, lon) in LOKASI.items():
    params = {"latitude": lat, "longitude": lon,
              "start_date": "2022-01-01", "end_date": "2025-12-31",
              "daily": "temperature_2m_mean,precipitation_sum",
              "timezone": "auto"}
    try:
        resp = polite_get(API_URL, params=params)     # status code ditangani di polite_get
    except (requests.HTTPError, RuntimeError) as e:
        print("Gagal untuk", prov_name, "->", e)
        raise

    # Simpan respons mentah apa adanya
    path = RAW_DIR / f"openmeteo_{prov_name.replace(' ', '_')}.json"
    path.write_text(resp.text, encoding="utf-8")

    data = resp.json()
    assert "daily" in data, f"Respons tidak berisi 'daily' untuk {prov_name}"
    respons[prov_name] = data
    rows_api.append(dict(sumber="Open-Meteo", metode="API", file=str(path), url=resp.url,
                         tanggal_akses=dt.date.today().isoformat(), lisensi=LISENSI_API,
                         ukuran_byte=path.stat().st_size, md5=md5_file(path)))

prov = catat_provenance(rows_api)
print(len(prov), "baris provenance")

[200] OK | https://archive-api.open-meteo.com/v1/archive?latitude=-8.65&longitude=115.22&st
[200] OK | https://archive-api.open-meteo.com/v1/archive?latitude=-7.8&longitude=110.36&sta
[200] OK | https://archive-api.open-meteo.com/v1/archive?latitude=-6.92&longitude=107.61&st
[200] OK | https://archive-api.open-meteo.com/v1/archive?latitude=3.59&longitude=98.67&star
[200] OK | https://archive-api.open-meteo.com/v1/archive?latitude=-5.15&longitude=119.41&st
10 baris provenance


In [61]:
# Normalisasi JSON -> tabel: respons berisi dict (kolom -> daftar nilai); dijadikan 1 baris per provinsi per hari
print("Kunci respons :", list(respons["BALI"].keys()))
print("Satuan        :", respons["BALI"].get("daily_units"))

frames_api = []
for prov_name, data in respons.items():
    d = pd.DataFrame(data["daily"]).rename(columns={"time": "tanggal"})
    d["tanggal"] = pd.to_datetime(d["tanggal"])
    d.insert(0, "provinsi", prov_name)
    frames_api.append(d)
df_api = pd.concat(frames_api, ignore_index=True)

assert len(df_api) >= 30, "Tabel API harus minimal 30 baris"
print("\nJumlah baris:", len(df_api))
print(df_api.groupby("provinsi").size())
print("Nilai kosong:\n", df_api.isna().sum())

# Atribusi dan sitasi (wajib CC BY 4.0)
ATRIBUSI_API = ("Weather data by Open-Meteo.com, lisensi CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). "
                "Data diolah: agregasi harian ke bulanan.")
SITASI_API = "Zippenfenig, P. (2023). Open-Meteo.com Weather API. Zenodo. https://doi.org/10.5281/ZENODO.7970649"
print("\n" + ATRIBUSI_API)
print(SITASI_API)
df_api.head()

Kunci respons : ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']
Satuan        : {'time': 'iso8601', 'temperature_2m_mean': '°C', 'precipitation_sum': 'mm'}

Jumlah baris: 7305
provinsi
BALI                1461
DI YOGYAKARTA       1461
JAWA BARAT          1461
SULAWESI SELATAN    1461
SUMATERA UTARA      1461
dtype: int64
Nilai kosong:
 provinsi               0
tanggal                0
temperature_2m_mean    0
precipitation_sum      0
dtype: int64

Weather data by Open-Meteo.com, lisensi CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). Data diolah: agregasi harian ke bulanan.
Zippenfenig, P. (2023). Open-Meteo.com Weather API. Zenodo. https://doi.org/10.5281/ZENODO.7970649


,provinsi,tanggal,temperature_2m_mean,precipitation_sum
0,BALI,2022-01-01,25.9,7.4
1,BALI,2022-01-02,25.4,16.2
2,BALI,2022-01-03,25.3,9.3
3,BALI,2022-01-04,25.8,6.5
4,BALI,2022-01-05,26.1,0.8


## Soal 4 - Scraping etis: periksa robots.txt, lalu ambil tabel
Halaman statis yang dipilih: **Wikipedia (id) - Daftar ibu kota provinsi di Indonesia**. Relevan dengan topik karena ibu kota provinsi dipakai sebagai titik koordinat cuaca di Soal 3. Urutan: (1) periksa dan simpan robots.txt sebagai bukti, (2) hanya bila diizinkan, ambil halaman, (3) parse tabel.

In [62]:
# 1) Periksa robots.txt halaman yang akan diambil
PAGE_URL = "https://id.wikipedia.org/wiki/Daftar_ibu_kota_provinsi_di_Indonesia"
p = urlparse(PAGE_URL)
ROBOTS_URL = f"{p.scheme}://{p.netloc}/robots.txt"

# Ambil robots.txt (timeout + User-Agent + jeda sudah di polite_get) dan simpan sebagai bukti
r = polite_get(ROBOTS_URL)
Path("docs/bukti/robots_id_wikipedia.txt").write_text(r.text, encoding="utf-8")

# Tanyakan ke parser apakah User-Agent kita boleh mengambil halaman ini
rp = robotparser.RobotFileParser()
rp.parse(r.text.splitlines())
boleh = rp.can_fetch(USER_AGENT, PAGE_URL)
crawl_delay = rp.crawl_delay(USER_AGENT)         # None bila situs tidak menetapkan
waktu_cek = dt.datetime.now().isoformat(timespec="seconds")

print("robots.txt        :", ROBOTS_URL)
print("User-Agent        :", USER_AGENT)
print("Halaman           :", PAGE_URL)
print("can_fetch         :", boleh)
print("crawl_delay       :", crawl_delay)
print("Jeda yang dipakai :", DELAY, "detik")
print("Waktu pemeriksaan :", waktu_cek)
print("\nCuplikan awal robots.txt:\n", r.text[:500])

assert boleh, "robots.txt TIDAK mengizinkan halaman ini. Jangan lakukan scraping; pilih halaman lain."
if crawl_delay:
    DELAY = max(DELAY, crawl_delay)              # patuhi crawl-delay bila ada

[200] OK | https://id.wikipedia.org/robots.txt
robots.txt        : https://id.wikipedia.org/robots.txt
User-Agent        : UTS-DataAcq-INSTIKI/1.0 (mahasiswa Informatika; humas@instiki.ac.id)
Halaman           : https://id.wikipedia.org/wiki/Daftar_ibu_kota_provinsi_di_Indonesia
can_fetch         : True
crawl_delay       : None
Jeda yang dipakai : 2 detik
Waktu pemeriksaan : 2026-11-17T11:44:40

Cuplikan awal robots.txt:
 ﻿# robots.txt for http://www.wikipedia.org/ and friends
#
# Please note: There are a lot of pages on this site, and there are
# some misbehaved spiders out there that go _way_ too fast. If you're
# irresponsible, your access to the site may be blocked.
#

# Observed spamming large amounts of https://en.wikipedia.org/?curid=NNNNNN
# and ignoring 429 ratelimit responses, claims to respect robots:
# http://mj12bot.com/
User-agent: MJ12bot
Disallow: /

# advertising-related bots:
User-agent: Mediapa


In [63]:
# 2) Ambil halaman (hanya karena robots.txt mengizinkan) dan simpan HTML mentah apa adanya
from bs4 import BeautifulSoup

page = polite_get(PAGE_URL)
RAW_HTML = RAW_DIR / "wikipedia_ibu_kota_provinsi.html"
RAW_HTML.write_text(page.text, encoding="utf-8")
prov = catat_provenance([dict(sumber="Wikipedia (id)", metode="scraping", file=str(RAW_HTML), url=PAGE_URL,
                              tanggal_akses=dt.date.today().isoformat(),
                              lisensi="CC BY-SA 4.0 (teks Wikipedia); robots.txt dicek: docs/bukti/robots_id_wikipedia.txt",
                              ukuran_byte=RAW_HTML.stat().st_size, md5=md5_file(RAW_HTML))])

# BeautifulSoup: judul halaman dan jumlah tabel wikitable
soup = BeautifulSoup(page.text, "html.parser")
print("Judul halaman         :", soup.title.get_text(strip=True) if soup.title else "-")
print("Jumlah tabel wikitable:", len(soup.find_all("table", class_="wikitable")))

# pd.read_html: ambil tabel yang memuat kata "Ibu"; bila tidak ada, ambil semua tabel
try:
    daftar_tabel = pd.read_html(StringIO(page.text), match="Ibu")
except ValueError:
    daftar_tabel = pd.read_html(StringIO(page.text))
print("Bentuk tabel ditemukan:", [t.shape for t in daftar_tabel])

# Pilih tabel terbesar (baris terbanyak) sebagai tabel utama
df_scrape = max(daftar_tabel, key=len).copy()

# Rapikan nama kolom (header Wikipedia kadang bertingkat)
if isinstance(df_scrape.columns, pd.MultiIndex):
    df_scrape.columns = [" ".join(dict.fromkeys(str(x) for x in c if "Unnamed" not in str(x))).strip()
                         for c in df_scrape.columns]
df_scrape.columns = [str(c).strip() for c in df_scrape.columns]

# Hapus penanda catatan kaki seperti [1] dari teks
def bersihkan(v):
    return re.sub(r"\[[^\]]*\]", "", v).strip() if isinstance(v, str) else v
for col in df_scrape.columns:
    df_scrape[col] = df_scrape[col].map(bersihkan)

assert len(df_scrape) >= 5, "Tabel hasil scraping terlalu kecil; periksa pemilihan tabel"
df_scrape.to_csv("data/processed/tabel_scrape_wikipedia.csv", index=False)
print("\nTabel hasil scraping:", df_scrape.shape)
print("Kolom:", list(df_scrape.columns))
df_scrape.head(10)

[200] OK | https://id.wikipedia.org/wiki/Daftar_ibu_kota_provinsi_di_Indonesia
Judul halaman         : Daftar ibu kota provinsi di Indonesia - Wikipedia bahasa Indonesia, ensiklopedia bebas
Jumlah tabel wikitable: 2
Bentuk tabel ditemukan: [(3, 2), (40, 7)]

Tabel hasil scraping: (40, 7)
Kolom: ['Provinsi[18]', 'Ibu Kota', 'Luas Wilayah (km2)', 'Jumlah Penduduk (2024)', 'Kepadatan (jiwa/km2)', 'IPM (2024)[19]', 'Ref']


,Provinsi[18],Ibu Kota,Luas Wilayah (km2),Jumlah Penduduk (2024),Kepadatan (jiwa/km2),IPM (2024)[19],Ref
0,Aceh,Banda Aceh,6136,265.310,4.324,8717.0,
1,Sumatera Utara,Medan,28199,2.486.283,8.817,8276.0,
2,Sumatera Barat,Padang,NaN,NaN,NaN,8399.0,
3,Riau,Pekanbaru,63226,1.167.599,1.847,8289.0,
4,Jambi,Jambi,NaN,NaN,NaN,8104.0,
5,Sumatera Selatan,Palembang,35251,1.801.367,5.111,8057.0,
6,Bengkulu,Bengkulu,NaN,NaN,NaN,8204.0,
7,Lampung,Bandar Lampung,NaN,NaN,NaN,7919.0,
8,Kepulauan Bangka Belitung,Pangkalpinang,NaN,NaN,NaN,8040.0,
9,Kepulauan Riau,Tanjungpinang,NaN,NaN,NaN,8063.0,NaN


In [64]:
# Keterkaitan sumber scraping dengan sumber lain: apakah ibu kota yang dipakai di Soal 3 ada di tabel Wikipedia?
ibu_kota = {"BALI": "Denpasar", "DI YOGYAKARTA": "Yogyakarta", "JAWA BARAT": "Bandung",
            "SUMATERA UTARA": "Medan", "SULAWESI SELATAN": "Makassar"}
teks_tabel = " ".join(df_scrape.astype(str).values.ravel()).lower()
for prov_name, kota in ibu_kota.items():
    print(f"{prov_name:18s} | ibu kota {kota:11s} | ada di tabel Wikipedia: {kota.lower() in teks_tabel}")

BALI               | ibu kota Denpasar    | ada di tabel Wikipedia: True
DI YOGYAKARTA      | ibu kota Yogyakarta  | ada di tabel Wikipedia: True
JAWA BARAT         | ibu kota Bandung     | ada di tabel Wikipedia: True
SUMATERA UTARA     | ibu kota Medan       | ada di tabel Wikipedia: True
SULAWESI SELATAN   | ibu kota Makassar    | ada di tabel Wikipedia: True


**Bukti kepatuhan scraping:** (1) `docs/bukti/robots_id_wikipedia.txt` berisi robots.txt yang diambil; (2) output `can_fetch` di atas menunjukkan hasil pemeriksaan untuk User-Agent dan halaman yang dipakai, beserta waktunya; (3) setiap request memakai timeout, User-Agent berisi email kampus, dan jeda; (4) HTML mentah tersimpan di `data/raw` dan tercatat di provenance (lisensi CC BY-SA 4.0).

## Soal 5 - Penggabungan data (berkas BPS + API Open-Meteo)
- **Kunci gabung:** `provinsi` + `periode` (format `YYYY-MM`).
- **BPS** berbentuk lebar (bulan sebagai kolom), diubah menjadi panjang: 1 baris = provinsi x bulan.
- **Open-Meteo** berbentuk harian, diagregasi menjadi bulanan: 1 baris = provinsi x bulan.
- Karena kedua sisi memiliki satu baris per kunci, dipakai `validate="one_to_one"`; bila ada kunci ganda, pandas akan menghentikan proses dengan error.

In [65]:
# Lima provinsi yang cuacanya diambil
lima_prov = list(LOKASI.keys())
BULAN_NO = {b: i + 1 for i, b in enumerate(BULAN)}

# --- Sumber BPS: bentuk lebar -> panjang (1 baris = provinsi x bulan)
bps_5 = df_all[df_all["provinsi"].isin(lima_prov)]
bps_long = bps_5.melt(id_vars=["provinsi", "tahun"], value_vars=BULAN,
                      var_name="bulan", value_name="perjalanan")
bps_long["periode"] = (bps_long["tahun"].astype(str) + "-" +
                       bps_long["bulan"].map(BULAN_NO).astype(str).str.zfill(2))
bps_long = bps_long[["provinsi", "periode", "perjalanan"]]
assert len(bps_long) == len(lima_prov) * 12 * df_all["tahun"].nunique(), "Jumlah baris BPS panjang tidak sesuai"
print("BPS (panjang):", bps_long.shape)

# --- Sumber API: harian -> bulanan (rata-rata suhu, total hujan)
df_api["periode"] = df_api["tanggal"].dt.strftime("%Y-%m")
cuaca_bulanan = (df_api.groupby(["provinsi", "periode"], as_index=False)
                 .agg(suhu_rata2_c=("temperature_2m_mean", "mean"),
                      hujan_total_mm=("precipitation_sum", "sum"),
                      hari_data=("tanggal", "count")))
print("Cuaca (bulanan):", cuaca_bulanan.shape)

BPS (panjang): (240, 3)
Cuaca (bulanan): (240, 5)


In [66]:
# Periksa kunci unik di kedua sisi sebelum merge
for nama, d in [("BPS", bps_long), ("Cuaca", cuaca_bulanan)]:
    ganda = d.duplicated(["provinsi", "periode"]).sum()
    print(f"{nama:6s}: {len(d)} baris | kunci ganda: {ganda}")

n_bps = len(bps_long)

# Merge dengan validate: one_to_one = tiap kunci muncul tepat sekali di kedua tabel
gabungan = bps_long.merge(cuaca_bulanan, on=["provinsi", "periode"],
                          how="left", validate="one_to_one", indicator=True)

# --- Bukti tidak ada duplikasi baris
print("\nBaris BPS sebelum merge :", n_bps)
print("Baris sesudah merge     :", len(gabungan))
print("Baris duplikat penuh    :", gabungan.duplicated().sum())
print("Kunci (provinsi+periode) ganda:", gabungan.duplicated(["provinsi", "periode"]).sum())
print("\nStatus pasangan:\n", gabungan["_merge"].value_counts())

assert len(gabungan) == n_bps, "Jumlah baris berubah setelah merge -> ada duplikasi"
assert gabungan.duplicated().sum() == 0
gabungan = gabungan.drop(columns="_merge")

# Gambaran awal (korelasi bukan bukti sebab-akibat)
print("\nKorelasi sederhana:")
print(gabungan[["perjalanan", "suhu_rata2_c", "hujan_total_mm"]].corr().round(2))
gabungan.head()

BPS   : 240 baris | kunci ganda: 0
Cuaca : 240 baris | kunci ganda: 0

Baris BPS sebelum merge : 240
Baris sesudah merge     : 240
Baris duplikat penuh    : 0
Kunci (provinsi+periode) ganda: 0

Status pasangan:
 _merge
both          240
left_only       0
right_only      0
Name: count, dtype: int64

Korelasi sederhana:
                perjalanan  suhu_rata2_c  hujan_total_mm
perjalanan            1.00         -0.76           -0.15
suhu_rata2_c         -0.76          1.00            0.02
hujan_total_mm       -0.15          0.02            1.00


,provinsi,periode,perjalanan,suhu_rata2_c,hujan_total_mm,hari_data
0,SUMATERA UTARA,2022-01,2022964.0,25.145161,237.3,31
1,JAWA BARAT,2022-01,9395919.0,22.545161,118.0,31
2,DI YOGYAKARTA,2022-01,2796450.0,25.335484,279.3,31
3,BALI,2022-01,991643.0,25.967742,295.2,31
4,SULAWESI SELATAN,2022-01,930906.0,26.064516,400.9,31


In [67]:
# Simpan hasil gabungan beserta metadata
OUT = Path("data/processed/data_gabungan.csv")
gabungan.to_csv(OUT, index=False)

metadata = {
    "dibuat": dt.datetime.now().isoformat(timespec="seconds"),
    "deskripsi": "Perjalanan wisatawan nusantara (BPS) digabung dengan cuaca bulanan (Open-Meteo) untuk lima provinsi tujuan",
    "sumber_digabung": ["BPS (berkas XLSX)", "Open-Meteo (API)"],
    "kunci_merge": ["provinsi", "periode (YYYY-MM)"],
    "how": "left", "validate": "one_to_one",
    "baris": int(len(gabungan)), "kolom": list(gabungan.columns),
    "baris_duplikat": int(gabungan.duplicated().sum()),
    "satuan": {"perjalanan": "perjalanan", "suhu_rata2_c": "derajat Celsius (rata-rata bulanan)",
               "hujan_total_mm": "milimeter (total bulanan)"},
    "md5_hasil": md5_file(OUT),
    "provenance": str(PROV_FILE),
    "atribusi": [SITASI_BPS, ATRIBUSI_API],
}
Path("data/processed/data_gabungan_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print("Tersimpan:", OUT, "dan data_gabungan_metadata.json")
metadata

Tersimpan: data/processed/data_gabungan.csv dan data_gabungan_metadata.json


{'dibuat': '2026-11-17T11:44:43',
 'deskripsi': 'Perjalanan wisatawan nusantara (BPS) digabung dengan cuaca bulanan (Open-Meteo) untuk lima provinsi tujuan',
 'sumber_digabung': ['BPS (berkas XLSX)', 'Open-Meteo (API)'],
 'kunci_merge': ['provinsi', 'periode (YYYY-MM)'],
 'how': 'left',
 'validate': 'one_to_one',
 'baris': 240,
 'kolom': ['provinsi',
  'periode',
  'perjalanan',
  'suhu_rata2_c',
  'hujan_total_mm',
  'hari_data'],
 'baris_duplikat': 0,
 'satuan': {'perjalanan': 'perjalanan',
  'suhu_rata2_c': 'derajat Celsius (rata-rata bulanan)',
  'hujan_total_mm': 'milimeter (total bulanan)'},
 'md5_hasil': 'e475278ab9c220caf2d20cdd4ff2361d',
 'provenance': 'docs/provenance.csv',
 'atribusi': ['Badan Pusat Statistik. Jumlah Perjalanan Wisatawan Nusantara Menurut Provinsi Tujuan (Perjalanan), 2022-2025. Diakses 12 November 2026, https://www.bps.go.id/id/statistics-table/2/MjIwMSMy/jumlah-perjalanan-wisatawan-nusantara-menurut-provinsi-tujuan-perjalanan.html',
  'Weather data by Open

## Audit akhir dan Data Ethics Checklist (Soal 2, lanjutan)
Audit otomatis memeriksa bukti utama sebelum checklist 9 butir diisi.

In [68]:
# Audit akhir
prov = pd.read_csv(PROV_FILE)

# 1) Provenance lengkap dan MD5 berkas mentah cocok dengan catatan
cek = pd.DataFrame([{"file": r["file"], "ada": Path(r["file"]).exists(),
                     "md5_sama": Path(r["file"]).exists() and md5_file(r["file"]) == r["md5"]}
                    for _, r in prov.iterrows()])
print(cek.to_string(index=False))
md5_ok = bool(cek["ada"].all() and cek["md5_sama"].all())

# 2) Syarat jumlah baris
baris_file_ok = len(df_all) >= 100
baris_api_ok = len(df_api) >= 30

# 3) Tidak ada domain terlarang pada seluruh URL sumber
terlarang = ["booking.com", "agoda", "traveloka", "tiket.com", "airbnb", "tripadvisor",
             "google.com/maps", "maps.google", "instagram", "facebook", "tiktok", "twitter", "x.com"]
semua_url = " ".join(prov["url"].astype(str).tolist() + [PAGE_URL, API_URL]).lower()
domain_terlarang = [d for d in terlarang if d in semua_url]

# 4) Tidak ada duplikasi pada hasil merge
tanpa_duplikat = bool(gabungan.duplicated().sum() == 0 and len(gabungan) == n_bps)

audit_ok = all([md5_ok, baris_file_ok, baris_api_ok, not domain_terlarang, tanpa_duplikat])
print("\nMD5 berkas mentah cocok  :", md5_ok)
print(f"Baris berkas >= 100      : {baris_file_ok} ({len(df_all)})")
print(f"Baris API >= 30          : {baris_api_ok} ({len(df_api)})")
print("Domain terlarang         :", domain_terlarang if domain_terlarang else "tidak ada")
print("Merge tanpa duplikasi    :", tanpa_duplikat)
print("\nAUDIT LULUS:", audit_ok)

                                                                             file  ada  md5_sama
data/raw/Jumlah Perjalanan Wisatawan Nusantara Menurut Provinsi Tujuan, 2022.xlsx True      True
data/raw/Jumlah Perjalanan Wisatawan Nusantara Menurut Provinsi Tujuan, 2023.xlsx True      True
data/raw/Jumlah Perjalanan Wisatawan Nusantara Menurut Provinsi Tujuan, 2024.xlsx True      True
data/raw/Jumlah Perjalanan Wisatawan Nusantara Menurut Provinsi Tujuan, 2025.xlsx True      True
                                                     data/raw/openmeteo_BALI.json True      True
                                            data/raw/openmeteo_DI_YOGYAKARTA.json True      True
                                               data/raw/openmeteo_JAWA_BARAT.json True      True
                                           data/raw/openmeteo_SUMATERA_UTARA.json True      True
                                         data/raw/openmeteo_SULAWESI_SELATAN.json True      True
                              

In [69]:
# Data Ethics Checklist (9 butir) dengan bukti, disimpan ke docs/
from IPython.display import Markdown, display

temuan_txt = "tidak ada kolom PII/quasi-identifier" if not temuan else str(temuan)
checklist = f"""# Data Ethics Checklist (9 butir)

| No | Butir | Status | Bukti |
|----|-------|--------|-------|
| 1 | Sumber data jelas dan resmi | Ya | BPS ({URL_BPS}); Open-Meteo (open-meteo.com); Wikipedia id (halaman statis). Rinci di docs/provenance.csv |
| 2 | Lisensi dataset diperiksa | Ya | BPS: Ketentuan Penggunaan (gratis, komersial/nonkomersial, wajib kutip) di docs/bukti/lisensi_bps.txt; Open-Meteo: CC BY 4.0, API gratis non-komersial di docs/bukti/lisensi_openmeteo.txt; Wikipedia: CC BY-SA 4.0 di docs/bukti/lisensi_wikipedia.txt |
| 3 | Ketentuan layanan portal dibaca | Ya | bps.go.id/assets/term-of-use dan open-meteo.com/en/terms, dibaca 2026-11-12 |
| 4 | Tidak ada PII | Ya | Pemindaian kolom (Soal 2): {temuan_txt} |
| 5 | Risiko quasi-identifier dinilai | Ya (rendah) | Data agregat provinsi x bulan; unit analisis wilayah, bukan individu |
| 6 | robots.txt diperiksa dan dipatuhi | Ya | docs/bukti/robots_id_wikipedia.txt; can_fetch={boleh} untuk halaman {PAGE_URL} pada {waktu_cek} |
| 7 | Timeout, User-Agent berisi email kampus, jeda antar-request | Ya | Fungsi polite_get: timeout {TIMEOUT} dtk, jeda {DELAY} dtk, retry untuk 429/5xx; User-Agent: {USER_AGENT} |
| 8 | Tidak memakai situs terlarang; tidak ada API key/token di notebook | Ya | Domain terlarang ditemukan: {domain_terlarang if domain_terlarang else "tidak ada"}; Open-Meteo tanpa key; email dibaca dari .env |
| 9 | Provenance lengkap, data mentah tidak diubah | {"Ya" if md5_ok else "PERIKSA"} | {len(prov)} baris provenance (URL, tanggal akses, lisensi, MD5); MD5 berkas mentah cocok: {md5_ok} |
"""
Path("docs/data_ethics_checklist.md").write_text(checklist, encoding="utf-8")
display(Markdown(checklist))

# Data Ethics Checklist (9 butir)

| No | Butir | Status | Bukti |
|----|-------|--------|-------|
| 1 | Sumber data jelas dan resmi | Ya | BPS (https://www.bps.go.id/id/statistics-table/2/MjIwMSMy/jumlah-perjalanan-wisatawan-nusantara-menurut-provinsi-tujuan-perjalanan.html); Open-Meteo (open-meteo.com); Wikipedia id (halaman statis). Rinci di docs/provenance.csv |
| 2 | Lisensi dataset diperiksa | Ya | BPS: Ketentuan Penggunaan (gratis, komersial/nonkomersial, wajib kutip) di docs/bukti/lisensi_bps.txt; Open-Meteo: CC BY 4.0, API gratis non-komersial di docs/bukti/lisensi_openmeteo.txt; Wikipedia: CC BY-SA 4.0 di docs/bukti/lisensi_wikipedia.txt |
| 3 | Ketentuan layanan portal dibaca | Ya | bps.go.id/assets/term-of-use dan open-meteo.com/en/terms, dibaca 2026-11-12 |
| 4 | Tidak ada PII | Ya | Pemindaian kolom (Soal 2): tidak ada kolom PII/quasi-identifier |
| 5 | Risiko quasi-identifier dinilai | Ya (rendah) | Data agregat provinsi x bulan; unit analisis wilayah, bukan individu |
| 6 | robots.txt diperiksa dan dipatuhi | Ya | docs/bukti/robots_id_wikipedia.txt; can_fetch=True untuk halaman https://id.wikipedia.org/wiki/Daftar_ibu_kota_provinsi_di_Indonesia pada 2026-11-17T11:44:40 |
| 7 | Timeout, User-Agent berisi email kampus, jeda antar-request | Ya | Fungsi polite_get: timeout 20 dtk, jeda 2 dtk, retry untuk 429/5xx; User-Agent: UTS-DataAcq-INSTIKI/1.0 (mahasiswa Informatika; humas@instiki.ac.id) |
| 8 | Tidak memakai situs terlarang; tidak ada API key/token di notebook | Ya | Domain terlarang ditemukan: tidak ada; Open-Meteo tanpa key; email dibaca dari .env |
| 9 | Provenance lengkap, data mentah tidak diubah | Ya | 10 baris provenance (URL, tanggal akses, lisensi, MD5); MD5 berkas mentah cocok: True |


In [70]:
# Kemas hasil (data + docs) dalam satu zip agar mudah diunduh/diunggah (.env tidak ikut)
import zipfile
with zipfile.ZipFile("hasil_uts_dam.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ["data", "docs"]:
        for fp in Path(folder).rglob("*"):
            if fp.is_file():
                z.write(fp)
print("Dibuat: hasil_uts_dam.zip |", round(Path("hasil_uts_dam.zip").stat().st_size / 1024, 1), "KB")

# Di Google Colab: unduh otomatis (diabaikan bila bukan di Colab)
try:
    from google.colab import files
    files.download("hasil_uts_dam.zip")
except ImportError:
    pass

Dibuat: hasil_uts_dam.zip | 131.6 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Ringkasan untuk presentasi
1. **Topik:** perjalanan wisatawan nusantara per provinsi (BPS) dan cuaca di provinsi tujuan.
2. **Tiga cara akuisisi:** berkas XLSX (BPS), API (Open-Meteo), scraping tabel (Wikipedia).
3. **Etika:** lisensi dicek dan dibuktikan; tidak ada PII; robots.txt diperiksa sebelum scraping; timeout, User-Agent berisi email kampus, jeda, dan retry; tanpa API key.
4. **Provenance:** URL, tanggal akses, lisensi, dan MD5 untuk setiap berkas mentah di `docs/provenance.csv`.
5. **Merge:** BPS (panjang) + cuaca bulanan, kunci provinsi + periode, `validate="one_to_one"`, jumlah baris tetap, nol duplikasi, hasil + metadata disimpan.
6. **Keterbatasan:** cuaca hanya di titik ibu kota; empat provinsi pemekaran Papua kosong di 2022; hubungan perjalanan-cuaca bersifat korelasi, bukan sebab-akibat.